# 11.4 - Vector Similarity

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

To retrieve by meaning you need a ruler for vector space. Cosine similarity measures angle, dot product measures alignment x magnitude, and Euclidean distance measures straight-line distance. Choosing the right metric (and normalizing correctly) is what makes retrieval work.

## 2. Why Does This Matter?

Retrieval quality depends on the metric. Cosine is the default for text; dot product is faster on normalized vectors; Euclidean matters for clustering. Get this wrong and long documents dominate or everything looks unrelated.

## 3. Prerequisites

Unit 11.3 (Embeddings).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement cosine, dot product, and Euclidean distance with numpy
- Prove that normalized vectors make dot product equal cosine
- Choose the right metric per use case from a decision table
- Compute pairwise similarity over embedded sentences and spot the most/least similar

## 5. Mental Model

Similarity metrics are different rulers for meaning-space: cosine measures angle, dot product measures alignment x magnitude, Euclidean measures distance.

```text
Cosine:  sim = (A.B) / (||A|| ||B||)      -> angle
Dot:     sim = A.B                         -> alignment x magnitude
Euclid:  dist = ||A - B||                  -> straight-line distance
```


## 6. Three Metrics from Scratch
Implement each with numpy so there is no black box.

In [1]:
import numpy as np


def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def dot_similarity(a, b):
    return np.dot(a, b)


def euclidean_distance(a, b):
    return np.linalg.norm(a - b)


A = np.array([0.1, 0.8, 0.3, 0.5])
B = np.array([0.2, 0.7, 0.4, 0.4])
C = np.array([0.9, 0.1, 0.1, 0.0])
print(f"cosine(A,B)={cosine_similarity(A,B):.4f}  dot(A,B)={dot_similarity(A,B):.4f}  euclid(A,B)={euclidean_distance(A,B):.4f}")
print(f"cosine(A,C)={cosine_similarity(A,C):.4f}  dot(A,C)={dot_similarity(A,C):.4f}  euclid(A,C)={euclidean_distance(A,C):.4f}")


cosine(A,B)=0.9811  dot(A,B)=0.9000  euclid(A,B)=0.2000
cosine(A,C)=0.2206  dot(A,C)=0.2000  euclid(A,C)=1.1916


## 7. Normalized Dot == Cosine
If both vectors have unit length, the denominator is 1 and dot product equals cosine. We verify numerically.

In [2]:
def unit(x):
    return x / np.linalg.norm(x)


Au, Bu = unit(A), unit(B)
print("dot(unitA, unitB)   :", round(float(np.dot(Au, Bu)), 6))
print("cosine(A, B)        :", round(float(cosine_similarity(A, B)), 6))
print("match?", np.isclose(np.dot(Au, Bu), cosine_similarity(A, B)))


dot(unitA, unitB)   : 0.981105
cosine(A, B)        : 0.981105
match? True


## 8. When to Use Which
Dot product on unnormalized vectors rewards magnitude - longer documents dominate. That is usually wrong for text retrieval, so we default to cosine (or normalize and use dot).

In [3]:
import pandas as pd
table = pd.DataFrame([
    ["Cosine", "Text retrieval, normalized embeddings", "Magnitude matters", "Default for RAG"],
    ["Dot product", "Embeddings normalized OR magnitude meaningful", "Different scales", "Fastest if normalized"],
    ["Euclidean", "Geometric clustering / k-means", "High-dim spaces", "Rare for text"],
], columns=["Metric", "Use when", "Avoid when", "Notes"])
print(table.to_string(index=False))


     Metric                                      Use when        Avoid when                 Notes
     Cosine         Text retrieval, normalized embeddings Magnitude matters       Default for RAG
Dot product Embeddings normalized OR magnitude meaningful  Different scales Fastest if normalized
  Euclidean                Geometric clustering / k-means   High-dim spaces         Rare for text


## 9. Pairwise Similarity Over Embedded Sentences
We embed a small corpus and find the closest pair (most similar meaning) and the farthest pair.

In [4]:
from numpy.random import default_rng
rng = default_rng(0)


def fake_embed(texts):
    # deterministic pseudo-embedding: same topic words -> closer vectors
    topics = {"return": np.array([1.0, .2, 0]), "shipping": np.array([.2, 1.0, 0]),
              "cook": np.array([0, .1, 1.0])}
    out = []
    for t in texts:
        v = np.zeros(3)
        for word, vec in topics.items():
            if word in t.lower():
                v = v + vec
        out.append(v if v.any() else rng.normal(size=3))
    return np.array(out)


texts = ["I want to return this item", "returns are allowed in 30 days",
         "how fast is shipping", "express shipping options",
         "recipe to cook pasta", "bake bread at home"]
vecs = fake_embed(texts)
V = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
sim = V @ V.T
n = len(texts)
best, worst = (0, 0, -1), (0, 0, 1)
for i in range(n):
    for j in range(i + 1, n):
        if sim[i, j] > best[2]:
            best = (i, j, sim[i, j])
        if sim[i, j] < worst[2]:
            worst = (i, j, sim[i, j])
print("most similar :", texts[best[0]], "<->", texts[best[1]], round(float(best[2]), 3))
print("least similar:", texts[worst[0]], "<->", texts[worst[1]], round(float(worst[2]), 3))


most similar : I want to return this item <-> returns are allowed in 30 days 1.0
least similar: how fast is shipping <-> bake bread at home -0.158


## 10. Similarity Heatmap
A matrix plot makes the clusters visible. On a semantic corpus we expect a clear block-diagonal structure (returns vs shipping vs cooking).

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
plt.imshow(sim, cmap="viridis", vmin=-1, vmax=1)
plt.colorbar(label="cosine similarity")
plt.xticks(range(n), [f"{i}" for i in range(n)])
plt.yticks(range(n), [f"{i}" for i in range(n)])
plt.title("Pairwise cosine similarity (fake embedding)")
plt.tight_layout()
plt.show()


C:\Users\PC\AppData\Local\Temp\ipykernel_7712\1711148702.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



## Common Mistakes

- Using dot product on unnormalized embeddings (long docs win).
- Mixing cosine and Euclidean in one system.
- Not normalizing before inner-product search.
- Assuming cosine is always in [0, 1] (it ranges -1 to 1).

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Irrelevant long docs rank highest | Dot product w/o normalization | Normalize or use cosine |
| Similar docs score low | Wrong metric for the model | Check model-documented metric |
| All similarities negative | Not normalized, opposite directions | Apply L2 normalization |

## Best Practices

- Default to cosine for text retrieval.
- L2-normalize before using dot-product search.
- Test metrics on real query-doc pairs.
- Document which metric your system uses.

## Hands-On Practice

1. **Basic:** Compute all three metrics for 3 vector pairs.
2. **Guided:** Embed 10 sentences and find most/least similar pairs.
3. **Independent:** Compare retrieval with cosine vs dot on the same index.
4. **Realistic:** Debug a system where dot product returns unexpected results.
5. **Challenge:** Implement a search function that accepts any metric as a parameter.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
